In [1]:
from pathlib import Path
import pandas as pd
import re
import subprocess

In [2]:
def obtain_lnL(path):
    mlc_tmp = open(path, 'r').read()
    lnL_tmp = re.search(r'\nlnL.*\n', mlc_tmp)
    try:
       lnL_tmp = lnL_tmp.group().split('):')[1].strip().split(' ')[0]
       return(lnL_tmp)
    except AttributeError:
        return(pd.NA) 
    #if lnL_tmp:
    #    return('Na')
    #else:
    # lnL_tmp = lnL_tmp.group().split('):')[1].strip().split(' ')[0]
    # return(lnL_tmp)
    
def parse_codeml_results(Ma_path, Man_path):
    Ma_lst = [i.parts[-2] for i in list(Path(Ma_path).rglob('mlc'))]
    Man_lst = [i.parts[-2] for i in list(Path(Man_path).rglob('mlc'))]
    
    Ma_lnL_lst = list()
    Man_lnL_lst = list()
    gene_lst = list()
    
    
    if Man_lst == Ma_lst:
        for num in range(0, len(Ma_lst)):
            Ma_mlc = Path(Ma_path) / Ma_lst[num] / 'mlc'
            Man_mlc = Path(Man_path) / Man_lst[num] / 'mlc'
            Ma_lnL = obtain_lnL(Ma_mlc)
            Man_lnL = obtain_lnL(Man_mlc)
            Ma_lnL_lst.append(Ma_lnL)
            Man_lnL_lst.append(Man_lnL)
            gene_lst.append(Ma_lst[num])
    
    return(pd.DataFrame({'Gene':gene_lst, 'Ma':Ma_lnL_lst, 'Man':Man_lnL_lst}))

def run_chi2_cmd(diff, df):
    cmd = f"/home/panda2bat/TOOLS/paml4.9j/bin/chi2 {df} {diff}"
    run = subprocess.Popen(cmd, shell=True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True)
    output = run.communicate()
    return_code = run.wait()
    return float(output[0].split('=')[-1].strip())

def chi2_test(dataframe, df):
    chi2_lst = list()
    gene_lst = list()
    for index, data in dataframe.iterrows():
        lnL_diff = abs(float(data.Ma) - float(data.Man))*2
        p = run_chi2_cmd(lnL_diff, df)
        chi2_lst.append(p)
        gene_lst.append(data.Gene)
        
    dataframe = pd.merge(dataframe, pd.DataFrame({'Gene': gene_lst,'chi2': chi2_lst}), on = 'Gene', how='inner')
    return(dataframe)

In [3]:
Ma_path = '/home/panda2bat/Avivorous_bat/output/14_evolution-selective/codeml/NycAvi/Ma'
Man_path = '/home/panda2bat/Avivorous_bat/output/14_evolution-selective/codeml/NycAvi/Man'
NycAvi_codeml = parse_codeml_results(Ma_path, Man_path)
NycAvi_codeml = NycAvi_codeml.dropna(axis = 0, how='any')
NycAvi_codeml = chi2_test(NycAvi_codeml, 1)

In [15]:
single2gene = pd.read_csv('/home/panda2bat/Avivorous_bat/output/11_evolution-single_copy_gene/orthofinder/output/Results_Jun19/orthologue2symbol.tsv', header = None, sep = '\t', names = ['Gene', 'SYMBOL'])

In [18]:
NycAvi_psg = NycAvi_codeml[NycAvi_codeml.chi2 < 0.05]
pd.merge(NycAvi_psg, single2gene, on='Gene', how='left').to_csv('/home/panda2bat/Avivorous_bat/statistic/positive_NycAvi/PSG.codeml.NycAvi.csv', sep = '\t', index = False)

In [17]:
NycAvi_psg

,Gene,Ma,Man,chi2
1,OG0005422,-5919.936350,-5942.522012,1.805000e-11
3,OG0008582,-5775.350617,-5778.597849,1.082000e-02
29,OG0003347,-724.864659,-727.438986,2.326000e-02
48,OG0006264,-18546.669394,-18550.133289,8.487000e-03
110,OG0007618,-7872.896248,-7875.169469,3.299000e-02
...,...,...,...,...
5496,OG0006453,-251.788822,-249.229665,2.367000e-02
5508,OG0007282,-4233.233013,-4245.819440,5.241000e-07
5519,OG0005311,-5081.444628,-5066.329940,3.839000e-08
5545,OG0008288,-8588.093204,-8590.813655,1.967000e-02


In [3]:
Ma_path = '/home/panda2bat/Avivorous_bat/output/14_evolution-selective/codeml/IaIo/Ma'
Man_path = '/home/panda2bat/Avivorous_bat/output/14_evolution-selective/codeml/IaIo/Man'
IaIo_codeml = parse_codeml_results(Ma_path, Man_path)
IaIo_codeml = IaIo_codeml.dropna(axis = 0, how='any')
IaIo_codeml = chi2_test(IaIo_codeml, 1)

In [4]:
IaIo_codeml

,Ma,Man,Gene,chi2
